# Cumulocity → SAP Business Data Cloud: data product

Publishes Cumulocity data of **one tenant** from this SAP Snowflake account to SAP Business Data Cloud (BDC) as a data product, so it can be installed in SAP Datasphere. Run `cumulocity_build_up.ipynb` for the same tenant first; this notebook reads its raw database and uses its warehouse and `OPS` schema.

| Object | Name | Purpose |
|---|---|---|
| Database (shared) | `CUMULOCITY_BDC_<T>` | Snowflake-managed Iceberg V3 tables, nothing else (one database per data product) |
| ↳ schema `CUMULOCITY` | `ALARM`, `INVENTORY`, `INVENTORY_*`, `EVENT`, `EVENT_*`, `OPERATION`, one table per measurement fragment, `MEASUREMENT` (all series, long format) | Dynamic Iceberg tables, the data product content |
| Share | `share_name` (`CUMULOCITY_ADHOC_DEMO`) | Snowflake share, associated with the BDC Zerocopy Connector; also the share name in BDC |
| Procedures, state | `CUMULOCITY_<T>.OPS.BDC_*` | Build, fingerprint, CSN, publish |
| Tasks | `OPS.BDC_CHECK`, `OPS.BDC_PUBLISH` | Run after `OPS.REBUILD_ON_SCHEMA_CHANGE`; republish only when the shared schema changes |

| Section | Does |
|---|---|
| 1. Parameters | Tenant, share, data product texts |
| 2. Database and share | Iceberg V3 database, schema, share |
| 3. Shared tables | `ALARM`, `INVENTORY` and the measurement tables, with primary keys |
| 4. Fingerprint and publish | State table, check, CSN and publish procedures |
| 5. Connect and publish | Share-back, share on the connector, first publish (changes BDC) |
| 6. Republish on schema change | Task graph |
| 7. Verification | Connector, publish log, task runs |

Remove the data product with `cumulocity_bdc_tear_down.ipynb`.

## 1. Parameters

`measurement_tables` selects the `cdc_measurement` tables of the data product (`LIKE` pattern, case-sensitive; `%` shares all, `c8y_SewPump%` only the sewage pumps). The texts go into the Open Resource Discovery (ORD) metadata that BDC shows in its catalog; they must not contain quotes. `OPS.BDC_PUBLISH` embeds them when it is created, so after changing a text, rerun `create_proc_bdc_publish` and `publish`.

In [ ]:
import re

catalog_name = ""                           # Cumulocity tenant ID, as in cumulocity_build_up.ipynb
share_name   = "CUMULOCITY_ADHOC_DEMO"       # Snowflake share = share and data product name in BDC
connector    = "CONNECTORS.ZEROCOPY.DEFAULT_SAP_BDC_CONNECTOR"   # SHOW ZEROCOPY CONNECTORS IN ACCOUNT
measurement_tables = "%"                    # cdc_measurement tables to share (LIKE), one shared table each
target_lag   = "10 minutes"                 # how far the shared tables may trail the Cumulocity data

title             = "Cumulocity adhoc-demo (Snowflake, eu-latest t2180777119)"
short_description = "Cumulocity alarms, events, operations, inventory and measurements"
description       = ("All Cumulocity data of the tenant from Streaming Lake Ingestion: current alarms, operations and inventory, "
                     "one table per inventory fragment, all events with one table per event fragment, and one table per "
                     "measurement fragment as in the Iceberg lake, and MEASUREMENT with every series in long format, refreshed every few minutes. "
                     "Join SOURCE of the measurement, event and alarm tables to INVENTORY.ID.")

assert re.fullmatch(r"[A-Za-z0-9]+", catalog_name), "catalog_name must be alphanumeric (used in object names)"
assert re.fullmatch(r"[A-Za-z0-9_]+", share_name), "share_name: letters, digits and underscores only"
for name, value in [("title", title), ("short_description", short_description), ("description", description),
                    ("measurement_tables", measurement_tables), ("target_lag", target_lag)]:
    assert "'" not in value and '"' not in value and "\\" not in value, f"{name} must not contain quotes or backslashes"

tenant  = catalog_name.upper()
raw_db  = f"CUMULOCITY_ICEBERG_{tenant}"
app_db  = f"CUMULOCITY_{tenant}"
wh      = f"CUMULOCITY_WH_{tenant}"
bdc_db  = f"CUMULOCITY_BDC_{tenant}"
share   = share_name

for label, value in [("Shared database", bdc_db), ("Share", share), ("Connector", connector),
                     ("Procedures, tasks", f"{app_db}.OPS"), ("Warehouse", wh)]:
    print(f"{label:20} {value}")

In [ ]:
%%sql -r set_role
USE ROLE ACCOUNTADMIN

In [ ]:
%%sql -r use_warehouse
USE WAREHOUSE {{wh}}

## 2. Database and share

BDC reads published tables through the connector, which accepts only Snowflake-managed Iceberg V3 tables with copy-on-write and the `COMPATIBLE` serialization policy. Setting these on the database makes every table in it inherit them. The data stays in Snowflake storage.

The share is created here, before the tables, so that the build procedure can grant each table to it. The share alone changes nothing in BDC; section 5 connects it.

In [ ]:
%%sql -r create_bdc_db
CREATE DATABASE IF NOT EXISTS {{bdc_db}}
  ICEBERG_VERSION_DEFAULT = 3
  CATALOG = 'SNOWFLAKE'
  ICEBERG_MERGE_ON_READ_BEHAVIOR = 'DISABLED'
  STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
  COMMENT = 'Cumulocity tenant {{tenant}}: data product {{share}} for SAP BDC (only shared tables)'

In [ ]:
%%sql -r create_bdc_schema
CREATE SCHEMA IF NOT EXISTS {{bdc_db}}.CUMULOCITY COMMENT = 'Shared with SAP BDC as data product {{share}}'

In [ ]:
%%sql -r create_share
CREATE SHARE IF NOT EXISTS {{share}} COMMENT = 'Cumulocity tenant {{tenant}}: data product for SAP BDC'

In [ ]:
%%sql -r grant_share_db
GRANT USAGE ON DATABASE {{bdc_db}} TO SHARE {{share}}

In [ ]:
%%sql -r grant_share_schema
GRANT USAGE ON SCHEMA {{bdc_db}}.CUMULOCITY TO SHARE {{share}}

## 3. Shared tables

Dynamic Iceberg tables, refreshed at most `target_lag` behind the source:

| Table | Source | One row per | Primary key |
|---|---|---|---|
| `ALARM` | `cdc_alarm."alarm"` | alarm, current state | `ID` |
| `INVENTORY` | `cdc_inventory."inventory"` | managed object, current state; deleted objects excluded | `ID` |
| `INVENTORY_<FRAGMENT>`, for example `INVENTORY_AGENT` | every other `cdc_inventory` table (`c8y_Agent`) | managed object with that fragment, current state | `ID` (= `INVENTORY.ID`) |
| `EVENT` | `cdc_event."event"` | event | `ID` |
| `EVENT_<FRAGMENT>`, for example `EVENT_WORKORDER` | every other `cdc_event` table (`c8y_WorkOrder`) | event with that fragment | `ID` (= `EVENT.ID`) |
| `OPERATION` | `cdc_operation."operation"` | operation, current state | `ID` |
| One per measurement fragment, for example `SEWPUMP_WINDINGTEMPC` | each `cdc_measurement` table matching `measurement_tables` (`c8y_SewPump_WindingTempC`) | measurement | `ID` |
| `MEASUREMENT` | every series of every `cdc_measurement` table matching `measurement_tables` | measurement and series | `ID`, `FRAGMENT`, `SERIES` |

Not shared: `cdc_rejected` (payloads the ingestion rejected), `internal` (ingestion state) and the `latest_*` tables, which Snowflake can't read (`Equality deletes on Iceberg tables are not supported`); `INVENTORY_*` carries the same current state, built from the change log.

The measurement tables keep the lake layout: one table per fragment, one column per series. The table name is the source name without `c8y_`, in upper case, as in the Databricks data product. The series object becomes two columns: the value as `DOUBLE` under the series name (`WINDING_TEMP_C`) and its unit (`WINDING_TEMP_C_UNIT`). `SOURCE` is the device (`INVENTORY.ID`).

`MEASUREMENT` holds the same values in long format, `ID, FRAGMENT, SERIES, TIME, SOURCE, VALUE, UNIT`, with the source names (`FRAGMENT` = `c8y_SewPump_WindingTempC`, `SERIES` = `winding_temp_c`). It is the fact for an Analytic Model in Datasphere and SAP Analytics Cloud, which can filter by fragment and series but can't switch tables; the Databricks product has the same table as `curated.curated_measurement`. A new fragment or series changes its query, so `BDC_CREATE_TABLES` replaces it, and the SAP side serves it again only after the delay described in the README.

Column names become `UPPER_SNAKE_CASE` (`lastUpdated` → `LAST_UPDATED`), timestamps are UTC `TIMESTAMP_NTZ(6)`, arrays and objects outside the measurement tables become JSON text. Change-log tables are reduced to the latest row per `ID`; rows whose latest change is a delete are left out. Every table takes every source column except `eventType`, so a new fragment adds a table and a new inventory fragment adds a column; both change the CSN, and section 6 republishes.

`OPS.BDC_CREATE_TABLES()` generates the tables from the current source columns (`CREATE OR REPLACE`, so it rebuilds them), adds the primary keys, grants them to the share and drops shared tables whose source is gone. It refuses to build when two sources map to the same table name: all tables share one schema, and Datasphere names them `<share>:v1.<TABLE>`. `OPS.BDC_CHECK()` calls it when the source columns change.

In [ ]:
%%sql -r create_proc_bdc_tables
CREATE OR REPLACE PROCEDURE {{app_db}}.OPS.BDC_CREATE_TABLES()
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
DECLARE
  props VARCHAR DEFAULT ' TARGET_LAG = ''{{target_lag}}'' WAREHOUSE = {{wh}} REFRESH_MODE = AUTO'
    || ' ICEBERG_VERSION = 3 CATALOG = ''SNOWFLAKE'' ICEBERG_MERGE_ON_READ_BEHAVIOR = ''DISABLED'''
    || ' STORAGE_SERIALIZATION_POLICY = ''COMPATIBLE''';
  built VARCHAR DEFAULT '|';
  created INTEGER DEFAULT 0;
  stale VARCHAR DEFAULT '';
  tag VARCHAR;
  same INTEGER;
  alias VARCHAR;
  note VARCHAR;
  alarm_tbl VARCHAR;
  alarm_latest_by VARCHAR;
  dupes VARCHAR;
  stmt VARCHAR;
BEGIN
  -- One shared table per source table. Select list: "col" AS "UPPER_SNAKE" (repeated underscores collapsed:
  -- BDC fails to extract a CSN element like ID__0), timestamps as UTC TIMESTAMP_NTZ(6);
  -- in measurement tables a series object becomes <SERIES> (DOUBLE) and <SERIES>_UNIT, elsewhere arrays and
  -- objects become JSON text (Iceberg tables in BDC need scalar columns)
  CREATE OR REPLACE TEMPORARY TABLE bdc_sources AS
  SELECT table_schema AS sch, table_name AS tbl,
         CASE WHEN table_schema = 'cdc_alarm' THEN 'ALARM'
              WHEN table_schema = 'cdc_inventory' AND table_name = 'inventory' THEN 'INVENTORY'
              WHEN table_schema = 'cdc_inventory' THEN 'INVENTORY_' || UPPER(REGEXP_REPLACE(table_name, '^c8y_', ''))
              WHEN table_schema = 'cdc_event' AND table_name = 'event' THEN 'EVENT'
              WHEN table_schema = 'cdc_event' THEN 'EVENT_' || UPPER(REGEXP_REPLACE(table_name, '^c8y_', ''))
              WHEN table_schema = 'cdc_operation' THEN 'OPERATION'
              ELSE UPPER(REGEXP_REPLACE(table_name, '^c8y_', '')) END AS alias,
         CASE WHEN table_schema = 'cdc_alarm' THEN 'Cumulocity alarms: current state per alarm. Key ID; SOURCE = INVENTORY.ID.'
              WHEN table_schema = 'cdc_inventory' AND table_name = 'inventory'
                THEN 'Cumulocity inventory: current state per managed object (devices, assets, groups). Key ID.'
              WHEN table_schema = 'cdc_inventory' AND table_name = 'c8y_ActiveAlarmsStatus'
                THEN 'Cumulocity uncleared (ACTIVE, ACKNOWLEDGED) alarms per severity per managed object with c8y_ActiveAlarmsStatus, counted from ALARM. Key ID = INVENTORY.ID.'
              WHEN table_schema = 'cdc_inventory' THEN 'Cumulocity inventory fragment ' || table_name || ': current state per managed object. Key ID = INVENTORY.ID.'
              WHEN table_schema = 'cdc_event' AND table_name = 'event' THEN 'Cumulocity events, all types. Key ID; SOURCE = INVENTORY.ID.'
              WHEN table_schema = 'cdc_event' THEN 'Cumulocity events, fragment ' || table_name || '. Key ID = EVENT.ID; SOURCE = INVENTORY.ID.'
              WHEN table_schema = 'cdc_operation' THEN 'Cumulocity operations: current state per operation. Key ID; DEVICE_ID = INVENTORY.ID.'
              ELSE 'Cumulocity measurements, fragment ' || table_name || '. Key ID; SOURCE = INVENTORY.ID.' END AS note,
         -- Latest row per ID; measurements are written once per ID and need no deduplication. Some updates
         -- keep lastUpdated (c8y_Availability changes only lastMessage), so the other timestamps break ties;
         -- Snowflake exposes no write order of Iceberg rows
         CASE WHEN table_schema = 'cdc_measurement' THEN NULL
              WHEN BOOLOR_AGG(column_name = 'lastUpdated')
                THEN '"lastUpdated" DESC'
                     || COALESCE(', ' || NULLIF(LISTAGG(IFF(data_type LIKE 'TIMESTAMP%' AND column_name NOT IN ('lastUpdated', 'creationTime'),
                                                           '"' || column_name || '" DESC NULLS LAST', NULL), ', ')
                                                  WITHIN GROUP (ORDER BY ordinal_position), ''), '')
                     || ', "eventType" DESC'
              ELSE '"time" DESC, "eventType" DESC' END AS latest_by,
         LISTAGG(CASE WHEN data_type LIKE 'TIMESTAMP%'
                        THEN 'CONVERT_TIMEZONE(''UTC'', "' || column_name || '")::TIMESTAMP_NTZ(6) AS "'
                             || REGEXP_REPLACE(UPPER(REGEXP_REPLACE(column_name, '([a-z0-9])([A-Z])', '\\1_\\2')), '_{2,}', '_') || '"'
                      WHEN data_type IN ('OBJECT', 'MAP') AND table_schema = 'cdc_measurement'
                        THEN '("' || column_name || '"::OBJECT):value::FLOAT AS "'
                             || REGEXP_REPLACE(UPPER(REGEXP_REPLACE(column_name, '([a-z0-9])([A-Z])', '\\1_\\2')), '_{2,}', '_') || '", '
                             || '("' || column_name || '"::OBJECT):unit::VARCHAR AS "'
                             || REGEXP_REPLACE(UPPER(REGEXP_REPLACE(column_name, '([a-z0-9])([A-Z])', '\\1_\\2')), '_{2,}', '_') || '_UNIT"'
                      WHEN data_type IN ('OBJECT', 'MAP')
                        THEN 'TO_JSON("' || column_name || '"::OBJECT) AS "'
                             || REGEXP_REPLACE(UPPER(REGEXP_REPLACE(column_name, '([a-z0-9])([A-Z])', '\\1_\\2')), '_{2,}', '_') || '"'
                      WHEN data_type = 'ARRAY'
                        THEN 'TO_JSON("' || column_name || '"::ARRAY) AS "'
                             || REGEXP_REPLACE(UPPER(REGEXP_REPLACE(column_name, '([a-z0-9])([A-Z])', '\\1_\\2')), '_{2,}', '_') || '"'
                      ELSE '"' || column_name || '" AS "'
                           || REGEXP_REPLACE(UPPER(REGEXP_REPLACE(column_name, '([a-z0-9])([A-Z])', '\\1_\\2')), '_{2,}', '_') || '"' END, ', ')
           WITHIN GROUP (ORDER BY ordinal_position) AS cols
  FROM {{raw_db}}.INFORMATION_SCHEMA.COLUMNS
  WHERE column_name <> 'eventType'
    AND (table_schema IN ('cdc_alarm', 'cdc_inventory', 'cdc_event', 'cdc_operation')
         OR (table_schema = 'cdc_measurement' AND table_name LIKE '{{measurement_tables}}'))
  GROUP BY table_schema, table_name
  -- MEASUREMENT: every series of every selected measurement table in long format, one row per measurement
  -- and series; cols holds the whole query. The fact for analytic models, which can't switch tables
  UNION ALL
  SELECT 'measurement_long', NULL, 'MEASUREMENT',
         'Cumulocity measurements, every fragment and series in long format: one row per measurement and series. Key ID, FRAGMENT, SERIES; SOURCE = INVENTORY.ID.',
         NULL,
         LISTAGG('SELECT "id" AS "ID", ''' || table_name || '''::VARCHAR AS "FRAGMENT", ''' || column_name || '''::VARCHAR AS "SERIES",'
                 || ' CONVERT_TIMEZONE(''UTC'', "time")::TIMESTAMP_NTZ(6) AS "TIME", "source" AS "SOURCE",'
                 || ' ("' || column_name || '"::OBJECT):value::FLOAT AS "VALUE", ("' || column_name || '"::OBJECT):unit::VARCHAR AS "UNIT"'
                 || ' FROM {{raw_db}}."cdc_measurement"."' || table_name || '" WHERE "id" IS NOT NULL AND "' || column_name || '" IS NOT NULL',
                 ' UNION ALL ') WITHIN GROUP (ORDER BY table_name, column_name)
  FROM {{raw_db}}.INFORMATION_SCHEMA.COLUMNS
  WHERE table_schema = 'cdc_measurement' AND table_name LIKE '{{measurement_tables}}' AND data_type IN ('OBJECT', 'MAP')
  HAVING COUNT(*) > 0;

  -- All tables share one schema, and Datasphere names them <share>:v1.<TABLE>: names must be unique
  SELECT LISTAGG(alias, ' ') INTO :dupes
  FROM (SELECT alias FROM bdc_sources GROUP BY alias HAVING COUNT(*) > 1);
  IF (dupes IS NOT NULL AND dupes <> '') THEN
    RETURN 'not built, duplicate table names: ' || dupes;
  END IF;

  -- Tables whose definition is unchanged stay as they are: a replaced table reaches Datasphere only hours later.
  -- The comment of each table ends with [<MD5 of its definition>]
  SHOW DYNAMIC TABLES IN SCHEMA {{bdc_db}}.CUMULOCITY;
  CREATE OR REPLACE TEMPORARY TABLE bdc_existing AS
    SELECT "name" AS name, "comment" AS note, "text" AS definition FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

  SELECT MAX(tbl), MAX(latest_by) INTO :alarm_tbl, :alarm_latest_by FROM bdc_sources WHERE sch = 'cdc_alarm';

  LET sources RESULTSET := (SELECT * FROM bdc_sources ORDER BY alias);
  FOR s IN sources DO
    IF (s.sch = 'measurement_long') THEN
      stmt := s.cols;
    ELSEIF (s.latest_by IS NULL) THEN
      stmt := 'SELECT ' || s.cols || ' FROM {{raw_db}}."' || s.sch || '"."' || s.tbl || '" WHERE "id" IS NOT NULL';
    ELSEIF (s.sch = 'cdc_inventory' AND s.tbl = 'c8y_ActiveAlarmsStatus' AND alarm_tbl IS NOT NULL) THEN
      -- Cumulocity rewrites the counters several times with the same lastUpdated, so the latest change-log row
      -- is ambiguous; count the uncleared alarms instead. Rows: managed objects that have the fragment
      -- Counts as NUMBER(38, 9), the type of the source columns: a Datasphere reinstall can't change the column
      -- types of an installed remote table ("MISMATCH IN DATA TYPE PRECISION/SCALE")
      stmt := 'SELECT i."id" AS "ID", CONVERT_TIMEZONE(''UTC'', GREATEST_IGNORE_NULLS(i."lastUpdated", a."lastUpdated"))::TIMESTAMP_NTZ(6) AS "LAST_UPDATED",'
              || ' COALESCE(a."warning", 0)::NUMBER(38, 9) AS "WARNING", COALESCE(a."minor", 0)::NUMBER(38, 9) AS "MINOR",'
              || ' COALESCE(a."major", 0)::NUMBER(38, 9) AS "MAJOR", COALESCE(a."critical", 0)::NUMBER(38, 9) AS "CRITICAL"'
              || ' FROM (SELECT "id", "lastUpdated" FROM (SELECT * FROM {{raw_db}}."cdc_inventory"."c8y_ActiveAlarmsStatus"'
              || ' WHERE "id" IS NOT NULL QUALIFY ROW_NUMBER() OVER (PARTITION BY "id" ORDER BY ' || s.latest_by || ') = 1)'
              || ' WHERE NOT ENDSWITH(COALESCE("eventType", ''''), ''_DELETE'')) i'
              || ' LEFT JOIN (SELECT "source", MAX("lastUpdated") AS "lastUpdated",'
              || ' COUNT_IF(uncleared AND "severity" = ''WARNING'') AS "warning", COUNT_IF(uncleared AND "severity" = ''MINOR'') AS "minor",'
              || ' COUNT_IF(uncleared AND "severity" = ''MAJOR'') AS "major", COUNT_IF(uncleared AND "severity" = ''CRITICAL'') AS "critical"'
              || ' FROM (SELECT *, "status" IN (''ACTIVE'', ''ACKNOWLEDGED'') AS uncleared FROM {{raw_db}}."cdc_alarm"."' || alarm_tbl || '"'
              || ' WHERE "id" IS NOT NULL QUALIFY ROW_NUMBER() OVER (PARTITION BY "id" ORDER BY ' || alarm_latest_by || ') = 1)'
              || ' WHERE NOT ENDSWITH(COALESCE("eventType", ''''), ''_DELETE'') GROUP BY "source") a ON a."source" = i."id"';
    ELSE
      -- Latest state per ID; deleted objects are filtered after picking the latest row
      stmt := 'SELECT ' || s.cols || ' FROM (SELECT * FROM {{raw_db}}."' || s.sch || '"."' || s.tbl || '"'
              || ' WHERE "id" IS NOT NULL QUALIFY ROW_NUMBER() OVER (PARTITION BY "id" ORDER BY ' || s.latest_by || ') = 1)'
              || ' WHERE NOT ENDSWITH(COALESCE("eventType", ''''), ''_DELETE'')';
    END IF;
    alias := s.alias;
    note := s.note;
    tag := MD5(props || note || stmt);
    SELECT COUNT(*) INTO :same FROM bdc_existing e WHERE e.name = :alias AND ENDSWITH(e.note, '[' || :tag || ']');
    IF (same = 0) THEN
      -- Built before the tags: same query and comment, only tag it
      SELECT COUNT(*) INTO :same FROM bdc_existing e
      WHERE e.name = :alias AND e.note = :note AND CONTAINS(e.definition, :stmt);
      IF (same > 0) THEN
        EXECUTE IMMEDIATE 'ALTER DYNAMIC TABLE {{bdc_db}}.CUMULOCITY."' || s.alias || '" SET COMMENT = '''
                          || s.note || ' [' || tag || ']''';
      END IF;
    END IF;
    IF (same = 0) THEN
      EXECUTE IMMEDIATE 'CREATE OR REPLACE DYNAMIC ICEBERG TABLE {{bdc_db}}.CUMULOCITY."' || s.alias || '"' || props
                        || ' COMMENT = ''' || s.note || ' [' || tag || ']'' AS ' || stmt;
      EXECUTE IMMEDIATE 'ALTER DYNAMIC TABLE {{bdc_db}}.CUMULOCITY."' || s.alias || '" ADD PRIMARY KEY ('
                        || IFF(s.sch = 'measurement_long', 'ID, FRAGMENT, SERIES', 'ID') || ')';
      -- Replaced tables lose their share grants
      EXECUTE IMMEDIATE 'GRANT SELECT ON TABLE {{bdc_db}}.CUMULOCITY."' || s.alias || '" TO SHARE {{share}}';
      created := created + 1;
    END IF;
    built := built || s.alias || '|';
  END FOR;

  -- Drop shared tables whose source is gone (or that an earlier layout created)
  SHOW DYNAMIC TABLES IN SCHEMA {{bdc_db}}.CUMULOCITY;
  LET old RESULTSET := (SELECT "name" AS name FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
                        WHERE NOT CONTAINS(:built, '|' || "name" || '|'));
  FOR o IN old DO
    EXECUTE IMMEDIATE 'DROP DYNAMIC TABLE {{bdc_db}}.CUMULOCITY."' || o.name || '"';
    stale := stale || ' ' || o.name;
  END FOR;
  RETURN (REGEXP_COUNT(built, '\\|') - 1) || ' tables, ' || created || ' created or replaced' || IFF(stale = '', '', '; dropped' || stale);
END;
$$

## 4. Fingerprint and publish

Two fingerprints, both MD5 over `INFORMATION_SCHEMA.COLUMNS`:

- **Source** (the columns of every `cdc_alarm`, `cdc_inventory`, `cdc_event` and `cdc_operation` table and of the `measurement_tables`): when it changes, `BDC_CHECK` rebuilds the shared tables.
- **Shared schema** (table, column, type, precision, scale, nullability of `CUMULOCITY_BDC_<T>.CUMULOCITY`): the content of the CSN. When it differs from the last publish, `BDC_CHECK` reports `changed` and `BDC_PUBLISH` republishes.

`OPS.BDC_STATE` keeps one row per build and publish. A new sewage pump fragment or inventory column does both: it rebuilds the tables and, because the CSN gains a table or column, republishes.

`OPS.BDC_CSN()` generates the CSN document from the shared tables (types like the Databricks SDK: `cds.String`, `cds.Decimal` for all numbers, `cds.Double`, `cds.Boolean`, `cds.Timestamp`; primary keys as `key`). `OPS.BDC_PUBLISH()` passes it with the ORD texts to `SYSTEM$SAP_PUBLISH_DATA_PRODUCT`.

In [ ]:
%%sql -r create_bdc_state
CREATE TABLE IF NOT EXISTS {{app_db}}.OPS.BDC_STATE (
  kind VARCHAR,            -- 'built' (source fingerprint) or 'published' (shared schema fingerprint)
  signature VARCHAR,
  at TIMESTAMP_LTZ,
  detail VARCHAR
)

In [ ]:
%%sql -r create_proc_bdc_check
CREATE OR REPLACE PROCEDURE {{app_db}}.OPS.BDC_CHECK()
RETURNS VARCHAR
LANGUAGE SQL
EXECUTE AS CALLER  -- SYSTEM$SET_RETURN_VALUE is refused in owner's-rights procedures
AS
$$
DECLARE
  source_sig VARCHAR;
  built_sig VARCHAR;
  schema_sig VARCHAR;
  published_sig VARCHAR;
  built VARCHAR DEFAULT '';
  result VARCHAR;
BEGIN
  SELECT MD5(LISTAGG(table_schema || '.' || table_name || '.' || column_name || ':' || data_type, ',')
             WITHIN GROUP (ORDER BY table_schema, table_name, column_name))
    INTO :source_sig
  FROM {{raw_db}}.INFORMATION_SCHEMA.COLUMNS
  WHERE table_schema IN ('cdc_alarm', 'cdc_inventory', 'cdc_event', 'cdc_operation')
     OR (table_schema = 'cdc_measurement' AND table_name LIKE '{{measurement_tables}}');

  SELECT MAX_BY(signature, at) INTO :built_sig FROM {{app_db}}.OPS.BDC_STATE WHERE kind = 'built';
  IF (source_sig IS DISTINCT FROM built_sig) THEN
    CALL {{app_db}}.OPS.BDC_CREATE_TABLES() INTO :built;
    INSERT INTO {{app_db}}.OPS.BDC_STATE VALUES ('built', :source_sig, CURRENT_TIMESTAMP(), :built);
    built := built || '; ';
  END IF;

  SELECT MD5(LISTAGG(table_name || '.' || column_name || ':' || data_type || '(' || COALESCE(numeric_precision::VARCHAR, '')
                     || ',' || COALESCE(numeric_scale::VARCHAR, '') || ')' || is_nullable, ',')
             WITHIN GROUP (ORDER BY table_name, column_name))
    INTO :schema_sig
  FROM {{bdc_db}}.INFORMATION_SCHEMA.COLUMNS
  WHERE table_schema = 'CUMULOCITY';

  SELECT MAX_BY(signature, at) INTO :published_sig FROM {{app_db}}.OPS.BDC_STATE WHERE kind = 'published';
  result := IFF(schema_sig IS DISTINCT FROM published_sig, 'changed', 'unchanged');

  -- Read by the BDC_PUBLISH task condition; fails outside a task run, which is fine
  BEGIN
    SELECT SYSTEM$SET_RETURN_VALUE(:result);
  EXCEPTION
    WHEN OTHER THEN NULL;
  END;
  RETURN built || result || ' (schema ' || LEFT(schema_sig, 12) || ', last published '
         || COALESCE(LEFT(published_sig, 12), 'never') || ')';
END;
$$

In [ ]:
%%sql -r create_proc_bdc_csn
CREATE OR REPLACE PROCEDURE {{app_db}}.OPS.BDC_CSN()
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
DECLARE
  csn VARCHAR;
BEGIN
  SHOW PRIMARY KEYS IN SCHEMA {{bdc_db}}.CUMULOCITY;
  CREATE OR REPLACE TEMPORARY TABLE bdc_keys AS
    SELECT "table_name" AS table_name, "column_name" AS column_name FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

  -- Built as text: element order follows the column order (OBJECT keys would be sorted)
  SELECT '{"csnInteropEffective":"1.0","$version":"2.0","i18n":{},'
         || '"meta":{"creator":"cumulocity_bdc_data_product.ipynb","flavor":"inferred"},'
         || '"definitions":{"CUMULOCITY":{"kind":"context"},'
         || LISTAGG('"CUMULOCITY.' || table_name || '":{"kind":"entity","elements":{' || elements || '}}', ',')
              WITHIN GROUP (ORDER BY table_name)
         || '}}'
    INTO :csn
  FROM (
    SELECT c.table_name,
           LISTAGG('"' || c.column_name || '":' || TO_JSON(OBJECT_CONSTRUCT(
               -- All NUMBER columns as cds.Decimal(p, s): the publish rejected cds.Integer64 for NUMBER(13, 0)
               -- ("does not match the provider type [decimal(13, 0)]", 2026-11-07)
               'type', CASE WHEN c.data_type = 'NUMBER' THEN 'cds.Decimal'
                            WHEN c.data_type = 'FLOAT' THEN 'cds.Double'
                            WHEN c.data_type = 'BOOLEAN' THEN 'cds.Boolean'
                            WHEN c.data_type LIKE 'TIMESTAMP%' THEN 'cds.Timestamp'
                            WHEN c.data_type = 'DATE' THEN 'cds.Date'
                            WHEN c.data_type = 'TIME' THEN 'cds.Time'
                            ELSE 'cds.String' END,
               'precision', IFF(c.data_type = 'NUMBER', c.numeric_precision, NULL),
               'scale', IFF(c.data_type = 'NUMBER', c.numeric_scale, NULL),
               'key', IFF(k.column_name IS NOT NULL, TRUE, NULL),
               'notNull', IFF(k.column_name IS NOT NULL, TRUE, NULL))), ',')
             WITHIN GROUP (ORDER BY c.ordinal_position) AS elements
    FROM {{bdc_db}}.INFORMATION_SCHEMA.COLUMNS c
    LEFT JOIN bdc_keys k ON k.table_name = c.table_name AND k.column_name = c.column_name
    WHERE c.table_schema = 'CUMULOCITY'
    GROUP BY c.table_name
  );
  LET parsed VARIANT := PARSE_JSON(csn);  -- fails here if the document isn't valid JSON
  RETURN csn;
END;
$$

In [ ]:
%%sql -r create_proc_bdc_publish
CREATE OR REPLACE PROCEDURE {{app_db}}.OPS.BDC_PUBLISH()
RETURNS VARCHAR
LANGUAGE SQL
EXECUTE AS CALLER  -- SYSTEM$SAP_PUBLISH_DATA_PRODUCT is refused in owner's-rights procedures
AS
$$
DECLARE
  ord VARCHAR DEFAULT '{"title":"{{title}}","shortDescription":"{{short_description}}","description":"{{description}}"}';
  csn VARCHAR;
  sig VARCHAR;
  stmt VARCHAR;
  res VARCHAR;
BEGIN
  CALL {{app_db}}.OPS.BDC_CSN() INTO :csn;
  SELECT MD5(LISTAGG(table_name || '.' || column_name || ':' || data_type || '(' || COALESCE(numeric_precision::VARCHAR, '')
                     || ',' || COALESCE(numeric_scale::VARCHAR, '') || ')' || is_nullable, ',')
             WITHIN GROUP (ORDER BY table_name, column_name))
    INTO :sig
  FROM {{bdc_db}}.INFORMATION_SCHEMA.COLUMNS
  WHERE table_schema = 'CUMULOCITY';

  stmt := 'SELECT SYSTEM$SAP_PUBLISH_DATA_PRODUCT(''{{connector}}'', ''{{share}}'', '''
          || REPLACE(ord, '''', '''''') || ''', ''' || REPLACE(csn, '''', '''''') || ''')';
  EXECUTE IMMEDIATE :stmt;
  SELECT $1::VARCHAR INTO :res FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));
  INSERT INTO {{app_db}}.OPS.BDC_STATE VALUES ('published', :sig, CURRENT_TIMESTAMP(), :res);
  RETURN res;
END;
$$

In [ ]:
%%sql -r initial_bdc_build
-- Builds the tables (no stored source fingerprint yet) and reports whether the schema needs a publish
CALL {{app_db}}.OPS.BDC_CHECK()

In [ ]:
%%sql -r bdc_tables_status
SHOW DYNAMIC TABLES IN SCHEMA {{bdc_db}}.CUMULOCITY
->> SELECT "name", "rows", "refresh_mode", "scheduling_state", "target_lag", "is_iceberg" FROM $1

In [ ]:
%%sql -r bdc_csn_preview
CALL {{app_db}}.OPS.BDC_CSN()

## 5. Connect and publish

**This section changes SAP BDC.** It enables share-back on the connector (Snowflake → BDC), associates the share with it, and publishes the data product. Afterwards the data product appears in the BDC cockpit under *Catalog & Marketplace*, and in Datasphere's catalog, where you request and install it (see the README, *Consuming the data product*).

The connector `DEFAULT_SAP_BDC_CONNECTOR` is created by SAP when the SAP Snowflake system joins the BDC formation; it must be in state `CONNECTED`.

In [ ]:
%%sql -r connector_status
DESC ZEROCOPY CONNECTOR {{connector}}

In [ ]:
%%sql -r enable_share_back
ALTER ZEROCOPY CONNECTOR {{connector}} SET SHARE_BACK = TRUE

In [ ]:
%%sql -r add_share
ALTER ZEROCOPY CONNECTOR {{connector}} ADD SHARE {{share}}

In [ ]:
%%sql -r publish
CALL {{app_db}}.OPS.BDC_PUBLISH()

## 6. Republish on schema change

The Snowflake counterpart of the Databricks `bdc_check` → `bdc_changed` → `bdc_publish` tasks, as a task graph below the existing schema-change task:

| Task | Runs | Does |
|---|---|---|
| `OPS.REBUILD_ON_SCHEMA_CHANGE` | every 15 min (root, from the build-up notebook) | Rebuilds the `CURATED` objects on source column changes |
| `OPS.BDC_CHECK` | after the root | Rebuilds the shared tables on source column changes; compares the shared schema with the last publish; sets the return value `changed` / `unchanged` |
| `OPS.BDC_PUBLISH` | after `BDC_CHECK`, only `WHEN` its return value is `changed` | CSN, publish, store the fingerprint |

All three are serverless. A check without changes reads only `INFORMATION_SCHEMA` and takes seconds. The root task is suspended while children are added and resumed with all dependents.

In [ ]:
%%sql -r suspend_root
ALTER TASK {{app_db}}.OPS.REBUILD_ON_SCHEMA_CHANGE SUSPEND

In [ ]:
%%sql -r create_task_check
CREATE OR REPLACE TASK {{app_db}}.OPS.BDC_CHECK
  USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE = 'XSMALL'
  COMMENT = 'Rebuilds the BDC tables on source changes; returns changed when the shared schema differs from the last publish'
  AFTER {{app_db}}.OPS.REBUILD_ON_SCHEMA_CHANGE
AS
  CALL {{app_db}}.OPS.BDC_CHECK()

In [ ]:
%%sql -r create_task_publish
CREATE OR REPLACE TASK {{app_db}}.OPS.BDC_PUBLISH
  USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE = 'XSMALL'
  COMMENT = 'Republishes data product {{share}} to SAP BDC'
  AFTER {{app_db}}.OPS.BDC_CHECK
  WHEN SYSTEM$GET_PREDECESSOR_RETURN_VALUE('BDC_CHECK') = 'changed'
AS
  CALL {{app_db}}.OPS.BDC_PUBLISH()

In [ ]:
%%sql -r resume_graph
SELECT SYSTEM$TASK_DEPENDENTS_ENABLE('{{app_db}}.OPS.REBUILD_ON_SCHEMA_CHANGE')

## 7. Verification

In [ ]:
%%sql -r connector_shares
-- Expect status CONNECTED, share_back true and the share in "shares"
DESC ZEROCOPY CONNECTOR {{connector}}

In [ ]:
%%sql -r bdc_state_log
SELECT kind, LEFT(signature, 12) AS signature, at, detail
FROM {{app_db}}.OPS.BDC_STATE
ORDER BY at DESC
LIMIT 20

In [ ]:
%%sql -r bdc_task_history
SELECT name, state, scheduled_time, return_value, error_message
FROM TABLE({{app_db}}.INFORMATION_SCHEMA.TASK_HISTORY(RESULT_LIMIT => 100))
WHERE database_name = '{{app_db}}' AND name IN ('REBUILD_ON_SCHEMA_CHANGE', 'BDC_CHECK', 'BDC_PUBLISH')
ORDER BY scheduled_time DESC, name
LIMIT 15

In [ ]:
%%sql -r bdc_sample
-- Hourly winding temperature per sewage pump over the last day
SELECT DATE_TRUNC('hour', m.TIME) AS hour, i.NAME AS device, ROUND(AVG(m.WINDING_TEMP_C), 2) AS winding_temp,
       ANY_VALUE(m.WINDING_TEMP_C_UNIT) AS unit, COUNT(*) AS n
FROM {{bdc_db}}.CUMULOCITY.SEWPUMP_WINDINGTEMPC m
JOIN {{bdc_db}}.CUMULOCITY.INVENTORY i ON i.ID = m.SOURCE
WHERE m.TIME >= DATEADD('day', -1, CONVERT_TIMEZONE('UTC', CURRENT_TIMESTAMP())::TIMESTAMP_NTZ)
GROUP BY ALL
ORDER BY hour DESC, device
LIMIT 50